# MIP Platform Backend Client - Getting Started

This notebook demonstrates how to discover the Python client API and run experiments.

**Note:** Authentication is configured automatically when you open this notebook from the Platform UI. If you see auth errors, sign in again and reload `/notebook` to refresh `mip_token`.

## API Discovery

Use this notebook as your command reference.

- Top-level exports: `configure`, `get_client`, `Experiment`, `Algorithm`, `DataModel`
- Jupyter introspection: `Experiment.create?`, `help(Experiment)`, `Algorithm.list?`, `DataModel.list?`
- Press `Tab` after `Experiment.` / `Algorithm.` / `DataModel.` for autocomplete.

In [1]:
import inspect
import platform_backend_client as pbc
from platform_backend_client import configure, Experiment, DataModel

# 1. Setup Connection
# No manual token needed. The client reads mip_token/.mip_token injected by the UI.
configure()

In [2]:
# 2. Show all available client methods with signatures
print('Top-level API:', ', '.join(pbc.__all__))

def show_api(cls):
    print(f"\n{cls.__name__}")
    for name, member in inspect.getmembers(cls):
        if name.startswith('_'):
            continue
        if callable(member):
            try:
                signature = inspect.signature(member)
            except (TypeError, ValueError):
                signature = '(...)'
            print(f"  - {cls.__name__}.{name}{signature}")

for cls in (Experiment, DataModel):
    show_api(cls)

print('\nTip: run Experiment.create? or help(Experiment) for detailed docs.')

Top-level API: configure, get_client, Experiment, Algorithm, DataModel, FederatedLogisticRegression, show_api, search_api, as_table

Experiment
  - Experiment.create(name, algorithm_name, data_model, datasets, x=None, y=None, filters=None, parameters=None, preprocessing=None, mip_version=None)
  - Experiment.delete(self)
  - Experiment.get(uuid)
  - Experiment.list(limit=10, offset=0)
  - Experiment.run_transient(name, algorithm_name, data_model, datasets, x=None, y=None, filters=None, parameters=None, preprocessing=None, mip_version=None, raise_on_failure=True)
  - Experiment.wait(self, timeout=None, poll_interval=2, raise_on_failure=True)

DataModel
  - DataModel.list()

Tip: run Experiment.create? or help(Experiment) for detailed docs.


In [3]:
# 3. List recent experiments (if any)
experiments = Experiment.list()
print(f"Found {len(experiments)} experiments.")
for exp in experiments[:5]:
    print(f"- {exp.name} ({exp.status})")

Found 0 experiments.


In [7]:
# 4. Run a new Experiment
# Let's run a simple descriptive statistics
try:
    new_exp = Experiment.create(
        name="My First Analysis",
        algorithm_name="pca",
        data_model="Stroke:3.7",
        datasets=["SFER"],
        y=["age", "acute_treat_evt_ia_rtpa_dose"],
        x=[]
    )
    print(f"Started experiment: {new_exp.uuid}")
    
    # Wait for results
    print("Waiting for results...")
    new_exp.wait()
    
    print("Results:")
    print(new_exp.results)

except Exception as e:
    print(f"Error running experiment: {e}")

Started experiment: bfe342f5-e961-43c0-8ce3-a036238e7e37
Waiting for results...
Results:
{'title': 'Eigenvalues and Eigenvectors', 'n_obs': 282.0, 'eigenvalues': [0.9999999999999929, 2.8500056618518546e-29], 'eigenvectors': [[1.0, 0.0], [1.1341630096416051e-30, 1.0]]}


In [11]:
data_models = DataModel.list()
data_models
print(data_models)

[<DataModel(code='Stroke', version='3.7')>, <DataModel(code='tbi', version='v9')>]
